# iGene VCF → FHIR Genomic Report `variant` Observations

This notebook converts `Input/VCF/igene_example_data.vcf` — a small example VCF covering the four
variant categories iGene reports (sequence variant, intragenic CNV, multigenic CNV, structural
variant) — into FHIR `Observation` resources conforming to the **HL7 Genomics Reporting IG**
[`variant`](https://build.fhir.org/ig/HL7/genomics-reporting/StructureDefinition-variant.html) profile
(`http://hl7.org/fhir/uv/genomics-reporting/StructureDefinition/variant`).

This is a separate, standalone demonstration of the international HL7 genomics-reporting IG — it is
**not** part of the NW-GMSA HL7 v2 ↔ FHIR pipeline used elsewhere in this repo.

The `##INFO`/`##FORMAT` meta-lines in the VCF header already cite the LOINC code each field maps to
(e.g. `GENE` → LOINC 48018-6, `HGVSC` → LOINC 51958-7/48004-6). This notebook parses those
citations directly out of the header and uses them to drive the component mapping, rather than
hard-coding the mapping separately from the data.

Component codes and value shapes below were cross-checked against the IG's own published examples
(`Observation-VariantExample2`, `Observation-ExampleGermlineDEL`, `Observation-ExampleGermlineCNV`,
`Observation-NTHL1-snv-var`, `Observation-EGFR-L858R-var`, `Observation-MSH2-del-var`). Where the IG's
example set doesn't give us a confirmed LOINC *answer* code (e.g. the exact answer-list code for
`Pathogenic` under clinical significance, or for a free-text cytoband string), we deliberately emit a
**text-only** `CodeableConcept` rather than guess a coded value — see the "Lookup tables" section.

## 1. Parse the VCF header → field-to-LOINC mapping

In [1]:
import json
import re
import subprocess
from pathlib import Path

import pandas as pd

VCF_PATH = Path("Input/DSS/VCF/igene_example_data.vcf")
OUTPUT_DIR = Path("Output/FHIR/GenomicsReporting")
RESULTS_DIR = Path("Results/FHIR/GenomicsReporting")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

VARIANT_PROFILE = "http://hl7.org/fhir/uv/genomics-reporting/StructureDefinition/variant"

In [2]:
def parse_vcf_header(path):
    """Parse ##INFO/##FORMAT meta-lines into {field_id: {"description", "loinc": [codes]}}."""
    meta_re = re.compile(r'^##(INFO|FORMAT)=<ID=([^,]+),.*?Description="([^"]*)"')
    loinc_re = re.compile(r"(\d{4,5}-\d)")
    fields = {}
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.rstrip("\n")
            if not line.startswith("##"):
                continue
            m = meta_re.match(line)
            if not m:
                continue
            _, field_id, description = m.groups()
            loinc_codes = loinc_re.findall(description) if "LOINC" in description else []
            fields[field_id] = {"description": description, "loinc": loinc_codes}
    return fields


FIELD_MAP = parse_vcf_header(VCF_PATH)
pd.DataFrame(
    [{"field": k, "loinc": ", ".join(v["loinc"]) or "-", "description": v["description"]} for k, v in FIELD_MAP.items()]
)

,field,loinc,description
0,VARTYPE,-,iGene custom-field variant category (Sequence ...
1,GENE,48018-6,Gene studied [ID] (LOINC 48018-6)
2,CYTOBAND,48001-2,Cytogenetic (chromosome) location (LOINC 48001-2)
3,HGVSC,"51958-7, 48004-6","Transcript reference sequence and DNA change, ..."
4,HGVSP,48005-3,"Amino acid change, p.HGVS (LOINC 48005-3)"
5,HGVSG,81290-9,"Genomic DNA change, g.HGVS, exactly as given i..."
6,CLASS,53037-8,Genetic variation clinical significance [Imp] ...
7,CLASSEVIDENCE,-,Free-text summary of evidence supporting the c...
8,INHERITANCE,-,Inheritance of the variant: Maternal/Paternal/...
9,SVTYPE,-,Type of structural variant


## 2. Parse the VCF data rows

In [3]:
def parse_vcf_records(path):
    records = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.rstrip("\n").rstrip("\r")
            if not line or line.startswith("#"):
                continue
            chrom, pos, vid, ref, alt, qual, filt, info, fmt, sample = line.split("\t")
            info_dict = {}
            for item in info.split(";"):
                if "=" in item:
                    k, v = item.split("=", 1)
                    info_dict[k] = v
                else:
                    info_dict[item] = True
            format_dict = dict(zip(fmt.split(":"), sample.split(":")))
            records.append(
                {
                    "chrom": chrom,
                    "pos": int(pos),
                    "id": vid,
                    "ref": ref,
                    "alt": alt,
                    "info": info_dict,
                    "format": format_dict,
                }
            )
    return records


RECORDS = parse_vcf_records(VCF_PATH)
pd.DataFrame(
    [
        {
            "ID": r["id"],
            "CHROM": r["chrom"],
            "POS": r["pos"],
            "REF": r["ref"],
            "ALT": r["alt"],
            "VARTYPE": r["info"].get("VARTYPE"),
            "GENE": r["info"].get("GENE"),
        }
        for r in RECORDS
    ]
)

,ID,CHROM,POS,REF,ALT,VARTYPE,GENE
0,SEQV1,17,41276046,TCT,T,Sequence_Variant,BRCA1
1,ICNV1,15,48797221,C,<DEL>,Intragenic_Copy_Number_Variant,FBN1
2,MCNV1,X,100652796,T,<DEL>,Multigenic_Copy_Number_Variant,None
3,SV1,X,100652796,T,<DEL>,Structural_Variant,None


## 3. Lookup tables

Coded values below (system + code) were confirmed against the IG's published `variant` examples.
Where the source data gives us free text that doesn't have a confirmed LOINC answer-list code
(`CLASS`, `CYTOBAND`), the component is built as **text-only** further down rather than asserting a
guessed code.

In [4]:
# GRCh37 primary-assembly RefSeq accessions for the chromosomes used in this VCF
# (VCF header: ##reference=GRCh37)
GRCH37_REFSEQ = {
    "17": "NC_000017.10",
    "15": "NC_000015.9",
    "X": "NC_000023.10",
}

# HGNC gene identifiers for the genes referenced in this VCF
HGNC_GENE = {
    "BRCA1": "HGNC:1100",
    "FBN1": "HGNC:3603",
}

# Sequence Ontology terms for "DNA change type" (LOINC 48019-4) - confirmed against
# Observation-NTHL1-snv-var (SNV), Observation-ExampleGermlineDEL (deletion) and
# Observation-ExampleGermlineCNV (copy_number_variation)
SO_TERM = {
    "SNV": ("SO:0001483", "SNV"),
    "deletion": ("SO:0000159", "deletion"),
    "insertion": ("SO:0000667", "insertion"),
    "copy_number_variation": ("SO:0001019", "copy_number_variation"),
}

# Allelic state (LOINC 53034-5) answer codes - confirmed against Observation-VariantExample2
# (heterozygous); homozygous/hemizygous follow the same LOINC answer-list (LL381-7) numbering.
ALLELIC_STATE = {
    "Heterozygous": ("LA6706-1", "heterozygous"),
    "Homozygous": ("LA6705-3", "homozygous"),
    "Hemizygous": ("LA6707-9", "hemizygous"),
}

# Origin of germline variant (LOINC 94186-4) answer code - confirmed against
# Observation-ExampleGermlineCNV. Only "Maternal" appears in this sample VCF, so that is
# the only value asserted with a code; anything else falls back to text-only.
INHERITANCE_ORIGIN = {
    "Maternal": ("LA26320-4", "Maternal"),
}

## 4. Build `variant` Observation resources

In [5]:
def cc(system=None, code=None, display=None, text=None):
    """Build a CodeableConcept, falling back to text-only when no coded value is asserted."""
    concept = {}
    if system and code:
        coding = {"system": system, "code": code}
        if display:
            coding["display"] = display
        concept["coding"] = [coding]
    if text:
        concept["text"] = text
    elif display and "coding" not in concept:
        concept["text"] = display
    return concept


def component(loinc_code, loinc_display, **value):
    comp = {"code": {"coding": [{"system": "http://loinc.org", "code": loinc_code, "display": loinc_display}]}}
    comp.update(value)
    return comp

In [6]:
def dna_change_type(record):
    """Map VARTYPE/SVTYPE/REF-ALT shape to a Sequence Ontology "DNA change type" term."""
    vartype = record["info"].get("VARTYPE", "")
    if "Copy_Number_Variant" in vartype:
        return SO_TERM["copy_number_variation"]
    if vartype == "Structural_Variant":
        return SO_TERM["deletion"] if record["info"].get("SVTYPE") == "DEL" else None
    ref, alt = record["ref"], record["alt"]
    if len(ref) == 1 and len(alt) == 1:
        return SO_TERM["SNV"]
    if len(alt) < len(ref):
        return SO_TERM["deletion"]
    if len(alt) > len(ref):
        return SO_TERM["insertion"]
    return None


def build_observation(record, obs_id, patient_ref, org_ref, effective_date):
    info = record["info"]
    fmt = record["format"]
    vartype = info.get("VARTYPE", "")
    is_structural = vartype in (
        "Intragenic_Copy_Number_Variant",
        "Multigenic_Copy_Number_Variant",
        "Structural_Variant",
    )

    components = []

    gene = info.get("GENE")
    if gene:
        hgnc = HGNC_GENE.get(gene)
        components.append(
            component(
                "48018-6",
                "Gene studied [ID]",
                valueCodeableConcept=cc("http://www.genenames.org", hgnc, gene) if hgnc else cc(text=gene),
            )
        )

    if "INHERITANCE" in info:
        components.append(
            component(
                "48002-0",
                "Genomic source class [Type]",
                valueCodeableConcept=cc("http://loinc.org", "LA6683-2", "Germline"),
            )
        )

    refseq = GRCH37_REFSEQ.get(record["chrom"])
    if refseq:
        components.append(
            component(
                "48013-7",
                "Genomic reference sequence [ID]",
                valueCodeableConcept=cc("http://www.ncbi.nlm.nih.gov/refseq", refseq),
            )
        )

    components.append(
        component(
            "92822-6",
            "Genomic coordinate system [Type]",
            valueCodeableConcept=cc("http://loinc.org", "LA30102-0", "1-based character counting"),
        )
    )

    components.append(component("69547-8", "Genomic ref allele [ID]", valueString=record["ref"]))
    components.append(component("69551-0", "Genomic alt allele [ID]", valueString=record["alt"]))

    so_term = dna_change_type(record)
    if so_term:
        code_val, display = so_term
        components.append(
            component(
                "48019-4",
                "DNA change type",
                valueCodeableConcept=cc("http://www.sequenceontology.org", code_val, display),
            )
        )

    hgvsc = info.get("HGVSC")
    if hgvsc:
        transcript_match = re.match(r"(N[MR]_\d+\.\d+)", hgvsc)
        if transcript_match:
            components.append(
                component(
                    "51958-7",
                    "Transcript reference sequence [ID]",
                    valueCodeableConcept=cc("http://www.ncbi.nlm.nih.gov/refseq", transcript_match.group(1)),
                )
            )
        components.append(
            component("48004-6", "DNA change (c.HGVS)", valueCodeableConcept=cc("http://varnomen.hgvs.org", hgvsc))
        )

    hgvsp = info.get("HGVSP")
    if hgvsp:
        components.append(
            component(
                "48005-3", "Amino acid change (pHGVS)", valueCodeableConcept=cc("http://varnomen.hgvs.org", hgvsp)
            )
        )

    hgvsg = info.get("HGVSG")
    if hgvsg:
        components.append(
            component(
                "81290-9", "Genomic DNA change (gHGVS)", valueCodeableConcept=cc("http://varnomen.hgvs.org", hgvsg)
            )
        )

    cytoband = info.get("CYTOBAND")
    if cytoband:
        components.append(
            component("48001-2", "Cytogenetic (chromosome) location", valueCodeableConcept=cc(text=cytoband))
        )

    classification = info.get("CLASS")
    if classification:
        components.append(
            component(
                "53037-8",
                "Genetic variation clinical significance [Imp]",
                valueCodeableConcept=cc(text=classification.replace("_", " ")),
            )
        )

    inheritance = info.get("INHERITANCE")
    if inheritance:
        origin = INHERITANCE_ORIGIN.get(inheritance)
        components.append(
            component(
                "94186-4",
                "Origin of germline genetic variant [Type]",
                valueCodeableConcept=cc("http://loinc.org", *origin) if origin else cc(text=inheritance),
            )
        )

    end = info.get("END")
    if is_structural and end:
        components.append(
            component(
                "81302-2",
                "Structural variant inner start and end",
                valueRange={"low": {"value": record["pos"]}, "high": {"value": int(end)}},
            )
        )
    elif not is_structural:
        components.append(
            component("81254-5", "Genomic allele start-end", valueRange={"low": {"value": record["pos"]}})
        )

    vaf = fmt.get("VAF")
    if vaf and vaf != ".":
        components.append(
            component(
                "81258-6",
                "Sample variant allelic frequency",
                valueQuantity={"value": float(vaf), "unit": "decimal", "system": "http://unitsofmeasure.org"},
            )
        )

    zyg = fmt.get("ZYG")
    if zyg:
        copy_match = re.search(r"\((\d+)_cop(?:y|ies)\)", zyg)
        if copy_match:
            components.append(
                component(
                    "82155-3",
                    "Genomic structural variant copy number",
                    valueQuantity={
                        "value": int(copy_match.group(1)),
                        "system": "http://unitsofmeasure.org",
                        "code": "1",
                    },
                )
            )
        elif zyg in ALLELIC_STATE:
            components.append(
                component("53034-5", "Allelic state", valueCodeableConcept=cc("http://loinc.org", *ALLELIC_STATE[zyg]))
            )

    return {
        "resourceType": "Observation",
        "id": obs_id,
        "meta": {"profile": [VARIANT_PROFILE]},
        "status": "final",
        "category": [
            {
                "coding": [
                    {"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "laboratory"}
                ]
            },
            {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0074", "code": "GE"}]},
        ],
        "code": {"coding": [{"system": "http://loinc.org", "code": "69548-6", "display": "Genetic variant assessment"}]},
        "subject": {"reference": patient_ref},
        "effectiveDateTime": effective_date,
        "performer": [{"reference": org_ref}],
        "valueCodeableConcept": cc("http://loinc.org", "LA9633-4", "Present"),
        "method": cc("http://loinc.org", "LA26398-0", "Sequencing"),
        "component": components,
    }

## 5. Assemble a `Bundle`

Subject is one of the [NW Genomics test patients](https://nw-gmsa.github.io/testing.html#integration-testing)
(`Input/PDS/NW.csv`), consistent with the rest of this repo's test data conventions, rather than an
invented NHS number.

In [7]:
patient = {
    "resourceType": "Patient",
    "id": "example-patient",
    "identifier": [{"system": "https://fhir.nhs.uk/Id/nhs-number", "value": "9737383192"}],
    "name": [{"family": "MANCHESTER", "given": ["Sansa"]}],
    "gender": "female",
    "birthDate": "1972-09-21",
}

organization = {
    "resourceType": "Organization",
    "id": "igene-laboratory",
    "name": "iGene Laboratory (example)",
}

EFFECTIVE_DATE = "2026-08-15"  # ##fileDate in the VCF header

observations = [
    build_observation(
        record,
        f"igene-{record['id'].lower()}",
        f"Patient/{patient['id']}",
        f"Organization/{organization['id']}",
        EFFECTIVE_DATE,
    )
    for record in RECORDS
]

bundle = {
    "resourceType": "Bundle",
    "id": "igene-example-data-variants",
    "type": "collection",
    "entry": (
        [{"fullUrl": f"urn:uuid:{patient['id']}", "resource": patient}]
        + [{"fullUrl": f"urn:uuid:{organization['id']}", "resource": organization}]
        + [{"fullUrl": f"urn:uuid:{o['id']}", "resource": o} for o in observations]
    ),
}

print(json.dumps(observations[0], indent=2))

{
  "resourceType": "Observation",
  "id": "igene-seqv1",
  "meta": {
    "profile": [
      "http://hl7.org/fhir/uv/genomics-reporting/StructureDefinition/variant"
    ]
  },
  "status": "final",
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/observation-category",
          "code": "laboratory"
        }
      ]
    },
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/v2-0074",
          "code": "GE"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "http://loinc.org",
        "code": "69548-6",
        "display": "Genetic variant assessment"
      }
    ]
  },
  "subject": {
    "reference": "Patient/example-patient"
  },
  "effectiveDateTime": "2026-08-15",
  "performer": [
    {
      "reference": "Organization/igene-laboratory"
    }
  ],
  "valueCodeableConcept": {
    "coding": [
      {
        "system": "http://loinc.org",
        "code": "

## 6. Save the FHIR output

In [8]:
bundle_path = OUTPUT_DIR / "igene_example_data.json"
with open(bundle_path, "w") as fh:
    json.dump(bundle, fh, indent=2)

for obs in observations:
    with open(OUTPUT_DIR / f"{obs['id']}.json", "w") as fh:
        json.dump(obs, fh, indent=2)

print(f"Wrote {bundle_path} and {len(observations)} individual Observation resources to {OUTPUT_DIR}/")

Wrote Output/FHIR/GenomicsReporting/igene_example_data.json and 4 individual Observation resources to Output/FHIR/GenomicsReporting/


## 7. Validate against the HL7 Genomics Reporting IG

Downloads the FHIR validator (reusing it if already present from `FHIR Validation.ipynb`) and checks
each `Observation` against the published IG package `hl7.fhir.uv.genomics-reporting#3.0.0`
(FHIR R4 / 4.0.1), same pattern as the NW-GMSA validation notebook. `-tx n/a` skips terminology-server
binding checks, since some components above are deliberately text-only.

In [9]:
from tqdm import tqdm
import requests

validator_jar = Path("validator_cli.jar")
if not validator_jar.exists():
    print("Downloading the FHIR Validator...")
    url = "https://github.com/hapifhir/org.hl7.fhir.core/releases/latest/download/validator_cli.jar"
    response = requests.get(url, stream=True)
    with open(validator_jar, "wb") as handle:
        for data in tqdm(response.iter_content(chunk_size=1024), unit="kB"):
            handle.write(data)
else:
    print("validator_cli.jar already present, reusing it.")

validator_cli.jar already present, reusing it.


In [10]:
for obs in observations:
    src = OUTPUT_DIR / f"{obs['id']}.json"
    out = RESULTS_DIR / f"{obs['id']}-OperationOutcome.json"
    subprocess.run(
        [
            "java", "-jar", str(validator_jar), str(src),
            "-version", "4.0.1",
            "-ig", "hl7.fhir.uv.genomics-reporting#3.0.0",
            "-profile", VARIANT_PROFILE,
            "-tx", "n/a",
            "-output", str(out),
            "-output-style", "json",
        ]
    )

FHIR Validation tool Version 6.9.9 (Git# f50ef63a178c). Built 2026-05-29T20:15:56.943Z (77 days old)


  Java:   21.0.7 from /Library/Java/JavaVirtualMachines/jdk-21.jdk/Contents/Home on x86_64 (64bit). 8192MB available
  Paths:  Current = /Users/kevinmayfield/github/MFT/Testing, Package Cache = /Users/kevinmayfield/.fhir/packages
  Params: Output/FHIR/GenomicsReporting/igene-seqv1.json -version 4.0.1 -ig hl7.fhir.uv.genomics-reporting#3.0.0 -profile http://hl7.org/fhir/uv/genomics-reporting/StructureDefinition/variant -tx n/a -output Results/FHIR/GenomicsReporting/igene-seqv1-OperationOutcome.json -output-style json
  Locale: United Kingdom/GB
  Jurisdiction: United Kingdom of Great Britain and Northern Ireland
Loading
  Loading FHIR v4.0.1 from hl7.fhir.r4.core#4.0.1


  Load hl7.terminology.r4#6.2.0 - 4288 resources (00:13.242)


  Load hl7.fhir.uv.extensions.r4#5.2.0 - 759 resources (00:02.723)
  Loaded FHIR - 8268 resources (00:00.000)
  Terminology server null - Version n/a: No Terminology Server (00:00.000)


  Load hl7.fhir.uv.extensions.r5#5.2.0 - 759 resources (00:01.175)


...

  Load hl7.fhir.uv.extensions#5.3.0 - 856 resources (00:00.412)


  Load hl7.terminology.r4#6.1.0 - 4279 resources (00:03.566)


  Load hl7.fhir.uv.extensions.r4#5.1.0 - 1397 resources (00:01.829)
  Load hl7.fhir.uv.genomics-reporting#3.0.0 - 93 resources (00:00.002)
  Package Summary: [hl7.fhir.r4.core#4.0.1, hl7.fhir.xver-extensions#0.1.0, hl7.terminology.r4#6.2.0, hl7.fhir.uv.extensions.r4#5.2.0, hl7.fhir.uv.extensions.r5#5.2.0, hl7.terminology.r5#7.1.0, hl7.fhir.uv.extensions.r5#5.3.0, hl7.terminology#7.3.0, hl7.fhir.uv.extensions#5.3.0, hl7.terminology.r4#6.1.0, hl7.fhir.uv.extensions.r4#5.1.0, hl7.fhir.uv.genomics-reporting#3.0.0]
  Terminology Cache at /var/folders/16/bplm70c55mj7020_2tms62sh0000gn/T/default-tx-cache
  Get set... 


  ...go! (00:08.167)
Cached new session. Cache size = 1
Sources to validate: Output/FHIR/GenomicsReporting/igene-sv1.json
Validating
  Profiles: [http://hl7.org/fhir/uv/genomics-reporting/StructureDefinition/variant]
  Validate Output/FHIR/GenomicsReporting/igene-sv1.json


 00:00.436
Done. Times: Loading: 00:33.467, validation: 00:00.436. Memory = 1Gb

Done. Times: Loading: 00:33.467, validation: 00:00.436. Max Memory = 8Gb


In [11]:
rows = []
for obs in observations:
    out = RESULTS_DIR / f"{obs['id']}-OperationOutcome.json"
    with open(out) as fh:
        oo = json.load(fh)
    for issue in oo.get("issue", []):
        rows.append(
            {
                "file": obs["id"],
                "severity": issue["severity"],
                "expression": ", ".join(issue.get("expression", [])),
                "details": issue.get("details", {}).get("text", ""),
            }
        )

df = pd.DataFrame(rows)
if not df.empty:
    df = df[~df["details"].str.contains("no terminology service", na=False)]
    df = df[~df["details"].str.contains("could not be found, so the code cannot be validated", na=False)]
df.sort_values(by=["file", "severity"]) if not df.empty else df

,file,severity,expression,details
22,igene-icnv1,information,Observation.component[10],This element does not match any known slice de...
28,igene-icnv1,information,Observation.component[2].value.ofType(Codeable...,Binding for path Observation.component[2].valu...
33,igene-icnv1,information,Observation.component[7].value.ofType(Codeable...,Binding for path Observation.component[7].valu...
40,igene-icnv1,warning,Observation.component[14].value.ofType(Quantity),Unable to validate code '1' in system 'http://...
41,igene-icnv1,warning,Observation,Constraint failed: dom-6: 'A resource should h...
42,igene-mcnv1,information,Observation.component[8],This element does not match any known slice de...
46,igene-mcnv1,information,Observation.component[1].value.ofType(Codeable...,Binding for path Observation.component[1].valu...
54,igene-mcnv1,warning,Observation.component[12].value.ofType(Quantity),Unable to validate code '1' in system 'http://...
55,igene-mcnv1,warning,Observation,Constraint failed: dom-6: 'A resource should h...
0,igene-seqv1,information,Observation.component[11],This element does not match any known slice de...
